# Pestaña 1 · Contexto y diseño experimental

In [1]:
import sys
sys.path.insert(0, "dashboard")
import pandas as pd
from IPython.display import HTML, Markdown, display
from src import figuras as fg
from src import textos as tx
from src.datos import num, objeto, pct, tabla

_primera = [True]


def figura(f):
    """Inserta la figura interactiva de Plotly (la librería se carga una vez por página)."""
    display(HTML(f.to_html(full_html=False, include_plotlyjs="cdn" if _primera[0] else False, config=fg.CONFIG,
                           default_width="100%")))
    _primera[0] = False


def texto(parrafos):
    display(Markdown("\n\n".join(parrafos)))


def tab(df, dec=3):
    display(df.style.format(precision=dec, decimal=",", thousands=" ", na_rep="—").hide(axis="index"))

La primera pestaña presenta el problema, la partición cronológica y el diseño experimental: cuántas combinaciones se entrenan y cómo se valida cada una sin sesgo de selección.

![Vista de la pestaña en el dashboard](../imagenes/pestana_1_contexto.png)

Las figuras son interactivas (se puede pasar el cursor sobre ellas) y las interpretaciones usan las mismas cifras que el dashboard y que el cuaderno.

## 1.1 Cifras del proyecto y esquema de validación anidada

In [2]:
c = objeto('contexto')
tab(pd.DataFrame({'Indicador': ['Llamadas tras eliminar duplicados', 'Entrenamiento', 'Prueba', 'Suscripción en entrenamiento', 'Suscripción en prueba', 'Combinaciones', 'Evaluaciones internas', 'Ajustes de modelos', 'Horas de CPU'], 'Valor': [num(c['registros_limpios']), f"{num(c['n_tr'])} ({c['desde_tr']} a {c['hasta_tr']})", f"{num(c['n_te'])} ({c['desde_te']} a {c['hasta_te']})", pct(c['prev_tr']), pct(c['prev_te']), str(c['combinaciones']), num(c['evaluaciones']), num(c['ajustes']), num(c['horas_cpu'], 1)]}))

Indicador,Valor
Llamadas tras eliminar duplicados,41 176
Entrenamiento,32 940 (2008-05 a 2009-05)
Prueba,8 236 (2009-05 a 2010-11)
Suscripción en entrenamiento,"6,4 %"
Suscripción en prueba,"30,8 %"
Combinaciones,140
Evaluaciones internas,13 730
Ajustes de modelos,41 890
Horas de CPU,"29,9"


In [3]:
figura(fg.esquema_anidado())

In [4]:
tab(tabla('desglose'), dec=0)
tab(tabla('esquema_anidado'), dec=1)

Rama,Modelos,Balanceo,Optimizadores,Operación,Combinaciones
Clasificación (guía),7,4,4,7 × 4 × 4,112
Regresión (guía),7,no aplica,4,7 × 4,28
Total,,,,,140


Pliegue externo,Registros de ajuste,Ajusta hasta,Valida desde,Valida hasta,Registros de validación,% positivos validación,Ajuste interno (mín.–máx.),Validación interna
1,4 983,2008-05,2008-05,2008-06,5 490,"3,9",741–3 231,1 245
2,10 473,2008-06,2008-06,2008-07,5 490,"6,0",2 112–7 348,2 618
3,15 963,2008-07,2008-07,2008-08,5 490,"5,9",3 486–11 466,3 990
4,21 453,2008-08,2008-08,2008-11,5 490,"5,4",4 857–15 583,5 363
5,26 943,2008-11,2008-11,2009-05,5 490,"14,0",6 231–19 701,6 735


In [5]:
texto(tx.contexto())

**Interpretación.** Tras eliminar duplicados quedan 41 176 llamadas: 32 940 de entrenamiento (2008-05 a 2009-05) y 8 236 de prueba (2009-05 a 2010-11). Suscribe el 6,4 % en entrenamiento y el 30,8 % en prueba. Se evaluaron 140 combinaciones en 5 pliegues externos, con 13 730 evaluaciones internas, 41 890 ajustes y 29,9 horas de CPU.

Esto significa que el modelo se juzga como se usaría: entrenado con el pasado y evaluado en campañas posteriores, donde la tasa de suscripción es casi cinco veces mayor. Ese cambio de prevalencia es tan importante como el desbalance de clases.

Por tal motivo, se opta por una validación anidada cronológica, por el AUC-PR como criterio principal y por analizar por separado el balanceo (que actúa sobre el umbral) y la recalibración mensual (que sigue el cambio de régimen).